#Creating silver layer and gold views for BI

In [0]:
USE CATALOG amiibo_data;
USE SCHEMA silver;

## Converting Columns to date type and dropping ingest_time 

In [0]:
CREATE OR REPLACE TABLE amiibo_converted_columns AS
SELECT 
    amiibo_series,
    character,
    game_series,
    image,
    name,
    CAST(release_au as DATE),
    CAST(release_eu as DATE),
    CAST(release_jp as DATE),
    CAST(release_na as DATE),
    tail,
    type
FROM amiibo_data.bronze.amiibo_all;

In [0]:
ALTER TABLE amiibo_converted_columns
ADD COLUMN release_global_earliest DATE;

UPDATE amiibo_converted_columns
SET release_global_earliest = LEAST(
    release_jp,
    release_na,
    release_eu,
    release_au
);

In [0]:
ALTER TABLE amiibo_converted_columns
ADD COLUMN release_global_latest DATE;

UPDATE amiibo_converted_columns
SET release_global_latest = GREATEST(
    release_jp,
    release_na,
    release_eu,
    release_au
);

In [0]:
SELECT * FROM amiibo_converted_columns

## Amiibos by game series View

In [0]:
CREATE OR REPLACE VIEW amiibo_data.gold.amiibos_by_game_series AS
SELECT
    game_series,
    COUNT(*) AS total_amiibos
FROM
    amiibo_converted_columns
GROUP BY
    game_series
ORDER BY
    total_amiibos DESC;

##Amiibos by amiibos series View

In [0]:
CREATE OR REPLACE VIEW amiibo_data.gold.amiibos_by_amiibo_series AS
SELECT 
  amiibo_series,
  COUNT(*) as amiibo_count
FROM amiibo_data.silver.amiibo_converted_columns
GROUP BY 
  amiibo_series
ORDER BY 
  amiibo_count DESC;

## Average date diff 

In [0]:
CREATE OR REPLACE VIEW amiibo_data.gold.avg_date_diff AS 
SELECT
    AVG(DATEDIFF(release_na, release_jp)) AS atraso_medio_na_dias,
    AVG(DATEDIFF(release_eu, release_jp)) AS atraso_medio_eu_dias,
    AVG(DATEDIFF(release_au, release_jp)) AS atraso_medio_au_dias
FROM
    amiibo_converted_columns
WHERE
    release_jp IS NOT NULL; 

## Realeses per month

In [0]:
CREATE OR REPLACE VIEW amiibo_data.gold.realese_per_month AS
SELECT
    DATE_FORMAT(release_global_earliest, 'yyyy-MM') AS ano_mes_lancamento,
    COUNT(*) AS total_amiibos_lancados
FROM
    amiibo_converted_columns
WHERE
    release_global_earliest IS NOT NULL
GROUP BY
    ano_mes_lancamento
ORDER BY
    total_amiibos_lancados DESC;

In [0]:
SELECT
  *
FROM amiibo_converted_columns
WHERE game_series = 'Animal Crossing' AND amiibo_series != 'Animal Crossing';

In [0]:
SELECT 
*
FROM amiibo_converted_columns
WHERE character = 'Mabel' AND name = 'Mabel' AND amiibo_series = 'Animal Crossing'